# Smoke Tube Visualisation

Visualise YOLO detections and smoke tubes across multiple sequences.
Each tube gets a unique colour; solid boxes = detections, dashed = gaps.

In [ ]:
from pathlib import Path

from smokeynet_adapted.data import (
    get_sorted_frames,
    is_wf_sequence,
    list_sequences,
    parse_timestamp,
)
from smokeynet_adapted.detector import load_model, run_yolo_on_frame
from smokeynet_adapted.tubes import (
    build_tubes,
    classify_tube_gt,
    draw_tubes_on_frames,
    plot_tube_grid,
    plot_tube_summary,
)
from smokeynet_adapted.types import FrameDetections

## Configuration

In [ ]:
DATA_DIR = Path("../data/01_raw/datasets/val")
MODEL_PATH = Path("../data/01_raw/models/best.pt")

# YOLO inference params
CONFIDENCE_THRESHOLD = 0.01
IOU_NMS = 0.2
IMAGE_SIZE = 1024
MAX_DETECTIONS_PER_FRAME = 10

# Tube construction params
TUBE_IOU_THRESHOLD = 0.2
TUBE_MAX_MISSES = 2

# How many sequences to visualise
NUM_SEQUENCES = 40

## Load model and discover sequences

In [ ]:
model = load_model(MODEL_PATH)
sequences = list_sequences(DATA_DIR)
print(f"Found {len(sequences)} sequences")
print("First 10:", [s.name for s in sequences[:10]])

## Helper: run detection + tube construction on a sequence

In [ ]:
def process_sequence(seq_dir):
    """Run YOLO + tube construction on a single sequence."""
    frame_paths = get_sorted_frames(seq_dir)
    gt = is_wf_sequence(seq_dir)

    frame_dets = []
    for idx, fpath in enumerate(frame_paths):
        dets = run_yolo_on_frame(
            model,
            fpath,
            conf=CONFIDENCE_THRESHOLD,
            iou_nms=IOU_NMS,
            img_size=IMAGE_SIZE,
        )
        if len(dets) > MAX_DETECTIONS_PER_FRAME:
            dets.sort(key=lambda d: d.confidence, reverse=True)
            dets = dets[:MAX_DETECTIONS_PER_FRAME]
        frame_dets.append(
            FrameDetections(
                frame_idx=idx,
                frame_id=fpath.stem,
                timestamp=parse_timestamp(fpath.stem),
                detections=dets,
            )
        )

    tubes = build_tubes(frame_dets, TUBE_IOU_THRESHOLD, TUBE_MAX_MISSES)

    return frame_paths, frame_dets, tubes, gt

## Visualise multiple sequences

For each sequence: frame grid with tube overlays, then combined
timeline + filmstrips view. Green border = smoke, red = false positive.

In [ ]:
import matplotlib.pyplot as plt

selected = sequences[:NUM_SEQUENCES]

for seq_dir in selected:
    frame_paths, frame_dets, tubes, gt = process_sequence(seq_dir)
    label = "WF" if gt else "FP"
    n_dets = sum(len(fd.detections) for fd in frame_dets)

    print(f"\n{'=' * 60}")
    print(f"{seq_dir.name} [{label}]")
    print(f"  Frames: {len(frame_paths)}, Detections: {n_dets}, Tubes: {len(tubes)}")

    # Frame grid with tube overlays
    annotated = draw_tubes_on_frames(frame_paths, tubes)
    frame_ids = [f"F{i}" for i in range(len(frame_paths))]
    fig = plot_tube_grid(annotated, frame_ids=frame_ids, cols=5)
    fig.suptitle(f"{seq_dir.name} [{label}]", fontsize=10, y=1.02)
    plt.show()

    # Tube summary (timeline + filmstrips)
    if tubes:
        frame_id_strs = [fp.stem for fp in frame_paths]
        tube_gt = [classify_tube_gt(tube, seq_dir, frame_id_strs) for tube in tubes]
        for tube, is_smoke in zip(tubes, tube_gt, strict=True):
            tag = "SMOKE" if is_smoke else "FP"
            start, end = tube.start_frame, tube.end_frame
            print(f"  Tube {tube.tube_id}: {tag} (frames {start}-{end})")

        fig = plot_tube_summary(
            frame_paths,
            tubes,
            num_frames=len(frame_paths),
            tube_labels=tube_gt,
            context_factor=1.2,
            title=f"{seq_dir.name} [{label}]",
        )
        plt.show()